# 🔵 Lab 03 — Embeddings & Vector Search
**Build a semantic search engine from scratch**

---
**Real-world scenario:** TomTom (Amsterdam) builds navigation software used by 600M+ devices.
Their map search must match 'gas station near highway' to 'petrol station beside A10 motorway' —
different words, same intent. Keyword search fails. Semantic embeddings solve it.

**What you will build:**
1. Generate sentence embeddings and understand what they represent
2. Measure semantic similarity with cosine distance
3. Prove embeddings beat keyword search on real examples
4. Visualise embedding clusters with t-SNE
5. Build a working FAQ semantic search engine

**Estimated time:** 55 min | **Level:** Intermediate | **No API key required**

In [ ]:
%pip install -q sentence-transformers numpy scikit-learn matplotlib umap-learn pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# BAAI/bge-small-en-v1.5: fast, excellent quality, 384-dim embeddings, runs on CPU
model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print(f'Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}')

## Part 1 — What is an Embedding?

An embedding is a list of ~384 numbers that encodes the *meaning* of a sentence.
Sentences with similar meanings produce embeddings that point in similar directions in 384-dimensional space.
We measure similarity with **cosine similarity**: 1.0 = identical direction, 0.0 = unrelated, -1.0 = opposite meaning.

In [ ]:
sentence_pairs = [
    ('Gas station near the highway', 'Petrol station beside the motorway'),      # same meaning, different words
    ('Navigate to Amsterdam Centraal', 'Get directions to Amsterdam train station'),  # same intent
    ('Turn left in 200 metres', 'The Eiffel Tower is in Paris'),                  # completely unrelated
    ('Traffic jam on A10', 'Congestion on the ring road'),                         # same situation
    ('Fastest route', 'Shortest path'),                                             # similar but not identical
    ('Fastest route', 'Avoid toll roads'),                                           # related but different intent
]

print(f'{"Sentence A":<45} {"Sentence B":<45} {"Similarity":>10}')
print('-' * 105)

for a, b in sentence_pairs:
    emb_a = model.encode(a, normalize_embeddings=True)
    emb_b = model.encode(b, normalize_embeddings=True)
    sim = float(np.dot(emb_a, emb_b))
    print(f'{a:<45} {b:<45} {sim:>10.3f}')

## Part 2 — Semantic vs Keyword Search

Keyword search (BM25 / TF-IDF) counts word overlap. It fails when user query and document use different words for the same concept. Embeddings find meaning — not word overlap.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TomTom POI (Points of Interest) database — 20 entries
poi_database = [
    'Shell petrol station, Rijksweg A10 Amsterdam',
    'BP fuel station, Schiphol highway exit',
    'McDonald\'s drive-through, ring road Amsterdam',
    'Charging station for electric vehicles, A4 motorway',
    'Hospital: Amsterdam UMC emergency ward',
    'Police station, Elandsgracht Amsterdam centrum',
    'Fire station, Amstelveenseweg',
    'Albert Heijn supermarket, Kalverstraat',
    'Parking garage, Museumplein P1',
    'Car park, Leidseplein underground',
    'Train station Amsterdam Centraal',
    'Metro station Weesperplein',
    'Bus terminal, Duivendrecht',
    'Hotel Hilton, Apollolaan Amsterdam',
    'Coffee shop (restaurant) Bagels & Beans',
    'Starbucks coffee, Rembrandtplein',
    'ATM machine, ING Bank, Dam Square',
    'Post office, PostNL Lijnbaansgracht',
    'Pharmacy, Apotheek Jordaan',
    'Bicycle repair shop, Waterlooplein',
]

# Test queries that use different words from the database
test_queries = [
    'gas station near highway',
    'place to park my car',
    'nearest hospital',
    'get some coffee',
    'charge my electric car',
]

# Embed the database
poi_embeddings = model.encode(poi_database, normalize_embeddings=True)

# TF-IDF for keyword search baseline
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(poi_database)

for query in test_queries:
    # Semantic search
    q_emb = model.encode(query, normalize_embeddings=True)
    sem_scores = np.dot(poi_embeddings, q_emb)
    sem_top = poi_database[np.argmax(sem_scores)]

    # Keyword search
    q_tfidf = tfidf.transform([query])
    kw_scores = (tfidf_matrix * q_tfidf.T).toarray().flatten()
    kw_top = poi_database[np.argmax(kw_scores)] if kw_scores.max() > 0 else '(no keyword match)'

    print(f'Query: "{query}"')
    print(f'  Semantic: {sem_top}')
    print(f'  Keyword:  {kw_top}')
    print()

## Part 3 — Visualising Embedding Space with t-SNE

Embeddings live in 384 dimensions — impossible to visualise directly.
t-SNE projects them into 2D while preserving neighbourhood structure.
Semantically similar sentences cluster together.

In [ ]:
# Sentences from 4 clearly distinct domains
domain_sentences = {
    'Navigation': [
        'Turn left in 300 metres', 'Take the motorway exit', 'Recalculating route',
        'You have reached your destination', 'Speed limit is 100 km/h',
    ],
    'Banking': [
        'Transfer money to account', 'Check bank balance', 'Apply for mortgage',
        'Credit card payment due', 'Interest rate fixed for 10 years',
    ],
    'Weather': [
        'Heavy rain expected tomorrow', 'Temperature will drop to 5 degrees',
        'Strong winds on the coast', 'Sunshine returning on Thursday', 'Fog warning on motorways',
    ],
    'Food': [
        'Restaurant serves Dutch cuisine', 'Best stroopwafel in Amsterdam',
        'Vegan menu available', 'Table reservation for two', 'Michelin star restaurant',
    ],
}

sentences, labels, colors_map = [], [], {'Navigation': '#00B4D8', 'Banking': '#E8A020', 'Weather': '#9B5DE5', 'Food': '#2EC4B6'}
for domain, sents in domain_sentences.items():
    sentences.extend(sents)
    labels.extend([domain] * len(sents))

embeddings = model.encode(sentences, normalize_embeddings=True)

# t-SNE projection to 2D
tsne = TSNE(n_components=2, perplexity=5, random_state=42, n_iter=1000)
coords = tsne.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0A1628')
ax.set_facecolor('#0A1628')

for domain, color in colors_map.items():
    mask = [l == domain for l in labels]
    x = coords[mask, 0]
    y = coords[mask, 1]
    ax.scatter(x, y, c=color, s=120, label=domain, zorder=5, alpha=0.9)
    for i, (xi, yi) in enumerate(zip(x, y)):
        domain_sents = domain_sentences[domain]
        ax.annotate(domain_sents[i][:25], (xi, yi), fontsize=7, color='#A0A8B0', xytext=(5, 5), textcoords='offset points')

ax.legend(framealpha=0.1, labelcolor='white')
ax.set_title('t-SNE Projection of Sentence Embeddings\nClusters emerge naturally by topic', color='#E8E4DC', fontsize=13)
ax.tick_params(colors='#4A5568')
for spine in ax.spines.values():
    spine.set_color('#1A2840')
plt.tight_layout()
plt.show()
print('Observation: sentences cluster by domain without any labels during training.')
print('The model learned semantic structure purely from language patterns.')

## Part 4 — Build a FAQ Search Engine

A realistic FAQ search for ING Bank — users ask questions in their own words, the engine finds the right answer.

In [ ]:
faq = [
    {'q': 'How do I reset my ING banking app PIN?', 'a': 'Open the ING app, go to Settings > Security > Change PIN. You will need to verify with your debit card and existing PIN first.'},
    {'q': 'What is the maximum daily ATM withdrawal limit?', 'a': 'The standard daily withdrawal limit at ATMs is €500. You can temporarily increase this to €2500 via the ING app under Card Settings.'},
    {'q': 'How long does an international transfer take?', 'a': 'SEPA transfers within the EU/EEA arrive the next business day. Transfers outside Europe take 2-5 business days.'},
    {'q': 'Can I open a joint bank account?', 'a': 'Yes, ING offers joint current accounts. Both account holders must be ING customers. Apply via the app or visit a branch.'},
    {'q': 'How do I cancel a direct debit?', 'a': 'Go to Payments > Direct Debits in the ING app. Select the direct debit and tap Cancel. This takes effect immediately.'},
    {'q': 'What documents do I need to open a business account?', 'a': 'You need a KvK (Chamber of Commerce) extract (max 3 months old), a valid ID, and proof of address dated within 90 days.'},
    {'q': 'How do I report a lost or stolen card?', 'a': 'Block your card immediately via the ING app (Card > Block). Call +31 20 22 888 88 to report theft and request a replacement.'},
    {'q': 'Is my savings account covered by deposit guarantee?', 'a': 'Yes, ING NL is covered by the Dutch Deposit Guarantee Scheme (DGS) up to €100,000 per person per institution.'},
]

# Index FAQ questions
faq_embeddings = model.encode([item['q'] for item in faq], normalize_embeddings=True)

def search_faq(user_query: str, top_k: int = 3) -> list:
    q_emb = model.encode(user_query, normalize_embeddings=True)
    scores = np.dot(faq_embeddings, q_emb)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(faq[i]['q'], faq[i]['a'], float(scores[i])) for i in top_indices]

# Test with queries that use different words from the FAQ
test_user_queries = [
    'forgot my app password',           # 'PIN' != 'password' but same intent
    'send money abroad',                 # 'international transfer' phrased differently
    'my wallet was stolen',              # 'lost or stolen card' phrased emotionally
    'start a business bank account',     # 'open a business account'
]

for query in test_user_queries:
    results = search_faq(query)
    print(f'User: "{query}"')
    print(f'  Best match (score={results[0][2]:.3f}): {results[0][0]}')
    print(f'  Answer: {results[0][1][:100]}...')
    print()

## ✅ Lab 03 Complete

You have:
- Generated sentence embeddings and measured cosine similarity between concepts
- Proved semantic search beats keyword search when user vocabulary differs from document vocabulary
- Visualised how t-SNE reveals natural topic clusters without any labels
- Built a working FAQ semantic search engine using only free, CPU-friendly libraries

**Next:** Lab 04 — RAG Pipeline